In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
sns.set_style('whitegrid')
sns.set_palette("Set2")

%matplotlib inline

# Leer los datos

In [3]:
totales = pd.read_csv("../../../data/respuestas_fede.csv")
print(totales.shape)

#globales
marmol = totales.loc[totales["escuela"] == "Colegio Modelo Mármol"]
mantovani = totales.loc[totales["escuela"] == "Escuela Nueva Juan Mantovani"]

(369, 22)


## Link de wpp vs Funcionalidad de Wikipedia

In [4]:
# Boilerplate para generar grafico de link wpp vs funcionalidad wiki
def link_wpp_vs_wiki(df, df_name, agrupando = False):
    link_wpp = df["link_descarga"]
    wiki_func = df["sobre_wikipedia"]

    def analizar_misc_link_wpp(val):
      if val == "Los mensajes recibidos no están verificados de ninguna forma. No podemos saber si el link es seguro.":
          return "Sin Misc."
      if val == "Los mensajes recibidos están verificados por personas en WhatsApp. El link es seguro.":
          return "Con Misc."
      if val == "No sé.":
          return "No sé"
      if val == "Si el mensaje fue enviado por uno de mis contactos entonces el link es seguro.":
          return "Con Misc."
      if val == "Los mensajes recibidos están verificados por un filtro inteligente. El link es seguro.":
          return "Con Misc."
    
    def analizar_misc_sobre_wiki(val):
      if val == "Hace falta investigar más para decidir si la información es verdadera o falsa.":
          return "Sin Misc."
      if val == "Toda la información en el artículo es verdadera porque está verificada.":
          return "Con Misc."
      if val == "No hay forma de saber si la información del artículo es verdadera o falsa.":
          return "Con Misc."
      if val == "Todo lo que está en el artículo es falso: cualquier usuario pudo haberlo inventado.":
          return "Con Misc."

    if(agrupando):
        filas = ["Sin Misc.",
                 "No sé",
                 "Con Misc."]
        columnas = ["Con Misc.",
                    "Sin Misc."]
        link_wpp = link_wpp.apply(analizar_misc_link_wpp)
        wiki_func = wiki_func.apply(analizar_misc_sobre_wiki)
        cross_tab_result = pd.crosstab(link_wpp, wiki_func)
        
    else:
        filas = ["Seguros.\n Verificados\n por personas",
                 "Seguros.\nVerificados\n por filtro\n inteligente", 
                 "Si es de mis\n contactos\n es seguro", 
                 "No sé", 
                 "No sabemos\n si es seguro"]
        columnas = ["Todo es\nfalso", 
                    "Inverificable", 
                    "Todo es\nverdad", 
                    "Hay que\ninvestigar\nmás"]
        cross_tab_result = pd.crosstab(link_wpp, wiki_func)
        cross_tab_result = cross_tab_result.rename(index={
                                    "Los mensajes recibidos no están verificados de ninguna forma. No podemos saber si el link es seguro.": "No sabemos\n si es seguro",
                                    "Los mensajes recibidos están verificados por personas en WhatsApp. El link es seguro.": "Seguros.\n Verificados\n por personas",
                                    "No sé.": "No sé",
                                    "Si el mensaje fue enviado por uno de mis contactos entonces el link es seguro.": "Si es de mis\n contactos\n es seguro",
                                    "Los mensajes recibidos están verificados por un filtro inteligente. El link es seguro.": "Seguros.\nVerificados\n por filtro\n inteligente"},
                                  columns={
                                    "Hace falta investigar más para decidir si la información es verdadera o falsa."     : "Hay que\ninvestigar\nmás",
                                    "Toda la información en el artículo es verdadera porque está verificada."            : "Todo es\nverdad",
                                    "No hay forma de saber si la información del artículo es verdadera o falsa."         : "Inverificable",
                                    "Todo lo que está en el artículo es falso: cualquier usuario pudo haberlo inventado.": "Todo es\nfalso"})

    # filas   = misconception de menor a mayor
    # columnas= misconception de mayor a menor
    cross_tab_result = cross_tab_result.reindex(filas, columns=columnas)


    def agregar_totales(fila):
       if(agrupando):
           return fila[0]+fila[1]
       else:
           return fila[0]+fila[1]+fila[2]+fila[3]
    
    def dividir(fila): 
       if(agrupando):
           return fila.div(fila[2])
       else:
           return fila.div(fila[4])

    cross_tab_result['totales'] = cross_tab_result.apply(agregar_totales, axis=1)
    cross_tab_result = cross_tab_result.apply(dividir, axis=1)
    cross_tab_result = cross_tab_result.drop(['totales'], axis=1)


    sns.heatmap(cross_tab_result, cmap='YlGnBu', annot=True, fmt='.2f', linewidths=0.5)
    plt.title('Relación de Respuestas\nLink wikipedia vs Funcionalidad Wikipedia\nDatos {} - Agrupando {}'.format(df_name,agrupando))
    plt.xlabel('Misconceptions Funcionalidad Wikipedia')
    plt.ylabel('Link wikipedia')
    # plt.show()
    plt.savefig('link_wpp_vs_wiki_{}_agrupando_{}.png'.format(df_name,agrupando))
    plt.clf()

In [5]:
# llamo al generador de graficos con los tres dfs
link_wpp_vs_wiki(totales, "totales")
link_wpp_vs_wiki(marmol, "marmol")
link_wpp_vs_wiki(mantovani, "mantovani")

# Agrupando misc
link_wpp_vs_wiki(totales, "totales", True)
link_wpp_vs_wiki(marmol, "marmol", True)
link_wpp_vs_wiki(mantovani, "mantovani", True)

<Figure size 640x480 with 0 Axes>